<a href="https://colab.research.google.com/github/SBethune103/virtual-running-coach-pipeline/blob/dev/notebooks/02_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q polars pyarrow

import polars as pl
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Paths now point to Google Drive
BASE = Path("/content/drive/MyDrive/virtual-running-coach")
DATA_RAW = BASE / "data/raw"
DATA_PROCESSED = BASE / "data/processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")
print("Raw data location:", DATA_RAW)
print("Processed data location:", DATA_PROCESSED)

✅ Setup complete


In [2]:
# Find weekly files
weekly_files = list(DATA_RAW.glob("run_ww_*_w.csv"))
print(f"Found {len(weekly_files)} weekly files")

if len(weekly_files) == 0:
    print("❌ No weekly files found. Please re-run the download cell in Notebook 1.")
else:
    # Load a sample (start with the first weekly file)
    df_weekly = pl.read_csv(
        weekly_files[0],
        try_parse_dates=True,
        null_values=["", "NA"]
    )

    print(f"Loaded: {df_weekly.shape[0]:,} rows × {df_weekly.shape[1]} columns")
    print(df_weekly.head())
    print("\nColumn types:")
    print(df_weekly.schema)

Found 0 weekly files


IndexError: list index out of range

In [ ]:
def clean_running_data(df: pl.DataFrame) -> pl.DataFrame:
    """Clean and engineer features for long-distance running data"""

    df = (
        df
        # Filter out zero-distance activities
        .filter(pl.col("distance") > 0)

        # Convert datetime if needed
        .with_columns(
            pl.col("datetime").str.to_datetime(strict=False).alias("date")
        )

        # Create useful features
        .with_columns([
            (pl.col("distance") / (pl.col("duration") / 60)).alias("avg_speed_kmh"),  # km/h
            (pl.col("duration") / pl.col("distance")).alias("pace_min_per_km"),       # min/km
            pl.col("date").dt.year().alias("year"),
            pl.col("date").dt.month().alias("month"),
            pl.col("date").dt.weekday().alias("weekday"),  # 1=Mon ... 7=Sun
        ])

        # Clean up
        .drop_nulls(subset=["distance", "duration"])
    )

    return df

df_clean = clean_running_data(df_weekly)

print(f"After cleaning: {df_clean.shape[0]:,} rows")
print(df_clean.select(["distance", "duration", "avg_speed_kmh", "pace_min_per_km"]).describe())

In [ ]:
print("=== Quick Insights ===")
print(f"\nTotal athletes: {df_clean['athlete'].n_unique():,}")
print(f"Average weekly distance: {df_clean['distance'].mean():.1f} km")
print(f"Median pace: {df_clean['pace_min_per_km'].median():.2f} min/km")

print("\nDistance distribution by age group:")
print(
    df_clean
    .group_by("age_group")
    .agg([
        pl.col("distance").mean().alias("avg_distance"),
        pl.col("athlete").n_unique().alias("athletes")
    ])
    .sort("avg_distance", descending=True)
)

In [ ]:
# Save as Parquet
output_path = DATA_PROCESSED / "running_weekly_clean.parquet"
df_clean.write_parquet(output_path)

print(f"✅ Clean data saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1_000_000:.2f} MB")

In [ ]:
olympic_file = DATA_RAW / "athlete_events.csv"

if olympic_file.exists():
    df_olympic = pl.read_csv(
        olympic_file,
        null_values=["NA", ""],
        infer_schema_length=10000
    )

    # Keep only Athletics (running events)
    df_running_oly = (
        df_olympic
        .filter(pl.col("Sport") == "Athletics")
        .filter(pl.col("Event").str.contains("(?i)metre|marathon|relay"))
        .select(["Name", "Sex", "Age", "Team", "Year", "Event", "Medal", "City"])
    )

    print(f"Olympic running events: {df_running_oly.shape[0]:,} rows")
    print(df_running_oly.head())

    # Save
    df_running_oly.write_parquet(DATA_PROCESSED / "olympic_running.parquet")
    print("✅ Olympic running data saved")